# Lorentzian Representation / Compression Error

(a) Fit each `S(\omega)` in the `dipole_polarizability_160Yb` dataset with 2, 3, ..., `N_MAX` Lorentzians and measure reconstruction error using strength-function-only objectives.

Below is a sweep cell given an input `N_MAX`. Results are cached under `tests/cache/yb_lorentzian_compression/`, so increasing `N_MAX` only fits the newly requested values.

# 1. Load EM1 run from runs_em1/

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "runs_em1").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root. Start Jupyter from the project root or notebooks/.")

REPO_ROOT = find_repo_root()
BETA_HELPER_DIR = REPO_ROOT / "Beta_decay_package" / "src"
if str(BETA_HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(BETA_HELPER_DIR))
import helper_gpt as beta_helper

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

dipole_helper = load_module(
    "dipole_helper_gpt_notebook",
    REPO_ROOT / "Dipole_polarizability" / "src" / "helper_gpt.py",
)

ALPHAD_FAC = 8.0 * np.pi * (7.29735e-3) * 197.33 / 9.0

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.is_file() else {}

def sorted_regex_params(match):
    groups = match.groupdict()
    if groups:
        return tuple(float(value) for _, value in sorted(groups.items(), key=lambda item: item[0]))
    return tuple(float(value) for value in match.groups())

def split_filename_params(fname):
    if not fname.startswith("strength_") or not fname.endswith(".out"):
        return None
    try:
        return tuple(float(part) for part in fname[len("strength_"):-len(".out")].split("_"))
    except ValueError:
        return None

def load_strength_dataset(data_dir, filename_regex=None):
    data_dir = Path(data_dir)
    pattern = re.compile(filename_regex) if filename_regex else None
    combined = []
    for fname in sorted(os.listdir(data_dir)):
        if pattern is None:
            params = split_filename_params(fname)
        else:
            match = pattern.match(fname)
            params = sorted_regex_params(match) if match else None
        if params is not None:
            combined.append((params, str(data_dir / fname)))
    if not combined:
        raise ValueError(f"No strength_*.out files found in {data_dir}")
    return combined

def rounded_key(values):
    return tuple(round(float(v), 4) for v in values)

def read_split_file(run_dir, data_dir, split, filename_regex=None):
    candidates = [run_dir / f"{split}_set.txt"]
    if split == "train":
        candidates.append(run_dir / "train_param_values.txt")
    if split == "test":
        candidates.append(run_dir / "test_param_values.txt")
    split_path = next((path for path in candidates if path.is_file()), None)
    if split_path is None:
        return []
    all_entries = load_strength_dataset(data_dir, filename_regex)
    by_key = {rounded_key(params): (params, path) for params, path in all_entries}
    entries = []
    for line in split_path.read_text().splitlines():
        if not line.strip():
            continue
        parts = line.split(",") if "," in line else line.split()
        key = rounded_key(float(part) for part in parts)
        entries.append(by_key[key])
    return entries

def infer_data_dir(run_dir, metadata, data_dir_override=None):
    if data_dir_override is not None:
        return Path(data_dir_override)
    meta_dir = metadata.get("strength_dir")
    if meta_dir and Path(meta_dir).is_dir():
        return Path(meta_dir)
    name = run_dir.name
    candidates = []
    if "160Yb" in name:
        candidates += [REPO_ROOT / "data" / "nuclear" / "160Yb_2d" / "total_strength"]
    if "48Ca" in name:
        candidates += [REPO_ROOT / "data" / "nuclear" / "48Ca_4d" / "total_strength_K0"]
    if "80Ni" in name:
        candidates += [REPO_ROOT / "data" / "nuclear" / "80Ni_2d" / "total_strength"]
    for candidate in candidates:
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("Could not infer data directory; set DATA_DIR manually.")

def infer_beta_n(num_params, num_components):
    for n in range(1, 500):
        expected = 2 * n + num_components * (n * (n + 1) // 2) + num_components + 2
        if expected == num_params:
            return n
    raise ValueError(f"Could not infer beta-style n from {num_params} parameters")

def default_params_file(run_dir):
    patterns = ["best_params_global.txt", "params_best_n*_retain*.txt", "params.txt", "**/params_n*_retain*_seed*.txt"]
    for pattern in patterns:
        matches = sorted(run_dir.glob(pattern))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No saved parameter file found below {run_dir}")

def load_run(run_dir, data_dir=None, params_path=None, model_family="auto"):
    run_dir = Path(run_dir)
    summary = load_json(run_dir / "run_summary.json")
    emulator = load_json(run_dir / "emulator.json")
    spec = emulator.get("spec", {})
    model = summary.get("config") or spec.get("model") or {}
    args = summary.get("args", {})
    metadata = {
        "summary": summary,
        "emulator": emulator,
        "model": model,
        "strength_dir": args.get("strength_dir") or spec.get("strength", {}).get("data_dir"),
        "strength_regex": args.get("strength_regex") or spec.get("strength", {}).get("filename_regex"),
        "central_point": summary.get("central_point") or spec.get("central_point"),
        "retain": model.get("retain"),
        "adapter": emulator.get("adapter"),
    }
    data_dir = infer_data_dir(run_dir, metadata, data_dir)
    combined = load_strength_dataset(data_dir, metadata.get("strength_regex"))
    params = np.loadtxt(Path(params_path) if params_path else default_params_file(run_dir))
    if model_family == "auto":
        model_family = "dipole" if ("Dipole" in str(metadata.get("adapter")) or (run_dir / "best_params_global.txt").is_file()) else "beta"
    splits = {name: read_split_file(run_dir, data_dir, name, metadata.get("strength_regex")) for name in ["train", "validation", "test"]}
    retain = float(metadata.get("retain") or 1.0)
    beta_n = None
    central_point = None
    coordinate_scales = None
    if model_family == "beta":
        num_components = len(combined[0][0])
        beta_n = infer_beta_n(len(params), num_components)
        points = np.asarray([entry[0] for entry in combined], dtype=float)
        central_point = tuple(metadata["central_point"]) if metadata.get("central_point") else combined[0][0]
        ranges = np.ptp(points, axis=0)
        coordinate_scales = tuple(float(v if v > 0 else 1.0) for v in ranges)
    return {
        "run_dir": run_dir,
        "data_dir": data_dir,
        "metadata": metadata,
        "model_family": model_family,
        "model": model,
        "combined": combined,
        "splits": splits,
        "params": params,
        "retain": retain,
        "beta_n": beta_n,
        "central_point": central_point,
        "coordinate_scales": coordinate_scales,
    }

# Choose any run under runs_em1/.
RUN_DIR = REPO_ROOT / "runs_em1" / "160Yb_2d_n13_randomfull_train121"
# RUN_DIR = REPO_ROOT / "runs_em1" / "48Ca_4d_K0_n30_affinew_localcluster_split"
DATA_DIR = None
MODEL_FAMILY = "auto"

run = load_run(RUN_DIR, data_dir=DATA_DIR, model_family=MODEL_FAMILY)
print("loaded", run["run_dir"])
print("model family:", run["model_family"])
print("data dir:", run["data_dir"])
print("points:", len(run["combined"]))
print("splits:", {k: len(v) for k, v in run["splits"].items()})


RuntimeError: Could not find repo root. Start Jupyter from the project root or notebooks/.

2) Evaluate the run at every available parameter point

In [ ]:
def strength_path(entry):
    return entry[1]

def alphaD_from_strength(x, y):
    return float(ALPHAD_FAC * np.trapezoid(y / np.maximum(x, 1.0e-6), x))

def m0_from_strength(x, y):
    return float(np.trapezoid(y, x))

def relative_l2(x, y_pred, y_true):
    return float(np.trapezoid((y_pred - y_true) ** 2, x) / (np.trapezoid(y_true ** 2, x) + 1.0e-16))

def predict_dipole(run, entry):
    model = run["model"]
    n = int(model["n"])
    num_params = int(model.get("n_params", len(entry[0])))
    config = dipole_helper.AnsatzConfig(
        n=n,
        n_params=num_params,
        ansatz=model.get("ansatz", "paper_dipole"),
        width_model=model.get("width_model", "affine"),
        use_vector_terms=bool(model.get("use_vector_terms", True)),
    )
    central_point = np.asarray(run["metadata"]["central_point"], dtype=np.float32)
    point = np.asarray([entry[0]], dtype=np.float32)
    M_batch, v_batch, fwhm_batch, _ = dipole_helper.build_model_matrices_and_vectors(
        tf.convert_to_tensor(run["params"], dtype=tf.float32),
        config,
        tf.convert_to_tensor(point, dtype=tf.float32),
        tf.convert_to_tensor(central_point, dtype=tf.float32),
    )
    eigenvalues, eigenvectors = tf.linalg.eigh(M_batch)
    eigenvalues = eigenvalues[0]
    eigenvectors = eigenvectors[0]
    k_keep = max(1, min(int(round(run["retain"] * int(eigenvalues.shape[0]))), int(eigenvalues.shape[0])))
    left = (int(eigenvalues.shape[0]) - k_keep) // 2
    right = left + k_keep
    poles = eigenvalues[left:right]
    vectors = eigenvectors[:, left:right]
    strengths = tf.square(tf.linalg.matvec(tf.transpose(vectors), v_batch[0]))
    true = np.loadtxt(strength_path(entry))
    x = tf.constant(true[:, 0], dtype=tf.float32)
    y_pred = dipole_helper.give_me_Lorentzian(x, poles, strengths, 0.5 * fwhm_batch[0]).numpy()
    return x.numpy(), y_pred, true[:, 1], poles.numpy(), strengths.numpy()

def predict_beta(run, entry):
    true = np.loadtxt(strength_path(entry))
    params_tf = tf.constant(run["params"], dtype=tf.float64)
    n = run["beta_n"]
    num_components = len(entry[0])
    D_mod, S_list, v0_mod, eta, width_params = beta_helper.modified_DS_general(params_tf, n, num_components)
    M = beta_helper.linear_matrix(D_mod, S_list, entry, run["central_point"], run["coordinate_scales"])
    eigenvalues, eigenvectors = tf.linalg.eigh(M)
    k_keep = max(1, min(int(round(run["retain"] * int(eigenvalues.shape[0]))), int(eigenvalues.shape[0])))
    left = (int(eigenvalues.shape[0]) - k_keep) // 2
    right = left + k_keep
    poles = eigenvalues[left:right]
    vectors = eigenvectors[:, left:right]
    strengths = tf.square(tf.linalg.matvec(tf.transpose(vectors), v0_mod))
    strengths = strengths * tf.cast((poles > 0) & (poles < 30), tf.float64)
    width = beta_helper.affine_width(eta, width_params, entry, run["central_point"], run["coordinate_scales"])
    x = tf.constant(true[:, 0], dtype=tf.float64)
    y_pred = beta_helper.give_me_Lorentzian(x, poles, strengths, width).numpy()
    return x.numpy(), y_pred, true[:, 1], poles.numpy(), strengths.numpy()

def predict_entry(run, entry):
    if run["model_family"] == "dipole":
        x, y_pred, y_true, poles, strengths = predict_dipole(run, entry)
        obs_true = alphaD_from_strength(x, y_true)
        obs_pred = alphaD_from_strength(x, y_pred)
        obs_name = r"$\alpha_D$"
    else:
        x, y_pred, y_true, poles, strengths = predict_beta(run, entry)
        obs_true = m0_from_strength(x, y_true)
        obs_pred = m0_from_strength(x, y_pred)
        obs_name = r"$m_0=\int S(E)dE$"
    return {
        "params": tuple(float(v) for v in entry[0]),
        "x": x,
        "y_pred": y_pred,
        "y_true": y_true,
        "poles": poles,
        "strengths": strengths,
        "observable_true": obs_true,
        "observable_pred": obs_pred,
        "observable_relerr": abs(obs_pred - obs_true) / max(abs(obs_true), 1.0e-12),
        "observable_name": obs_name,
        "strength_rel_l2": relative_l2(x, y_pred, y_true),
    }

MAX_EVAL_POINTS = None  # Set to an integer, e.g. 50, for faster exploratory runs.
eval_entries = run["combined"] if MAX_EVAL_POINTS is None else run["combined"][:MAX_EVAL_POINTS]
results = [predict_entry(run, entry) for entry in eval_entries]
points = np.asarray([row["params"] for row in results], dtype=float)
obs_relerr = np.asarray([row["observable_relerr"] for row in results], dtype=float)
print("evaluated", len(results), "points")
print("median observable relerr:", np.median(obs_relerr))
print("max observable relerr:", np.max(obs_relerr))


3) parameter grid/slice colored by observable relative error, plus chosen strength curve

In [ ]:
CHOSEN_INDEX = 0
CHOSEN_PARAMS = None  # Example: (1.1, 2.75). If set, nearest sampled point is used.
VARIED_COLS = (0, 1)  # For >2D, plot these two columns and hold the others near the chosen point.
N_SLICE = 30

def choose_index(points, chosen_index=0, chosen_params=None):
    if chosen_params is None:
        return int(chosen_index)
    target = np.asarray(chosen_params, dtype=float)
    return int(np.argmin(np.sum((points - target[None, :]) ** 2, axis=1)))

def nearest_slice(points, varied_cols, center, n_slice):
    held_cols = [col for col in range(points.shape[1]) if col not in varied_cols]
    if not held_cols:
        return np.arange(len(points))
    ranges = np.ptp(points[:, held_cols], axis=0)
    ranges = np.where(ranges > 0.0, ranges, 1.0)
    dist = np.linalg.norm((points[:, held_cols] - center[held_cols][None, :]) / ranges[None, :], axis=1)
    return np.argsort(dist)[:min(n_slice, len(points))]

chosen_idx = choose_index(points, CHOSEN_INDEX, CHOSEN_PARAMS)
chosen = results[chosen_idx]
center = points[chosen_idx]

fig, (ax_grid, ax_strength) = plt.subplots(1, 2, figsize=(13, 5.2))
if points.shape[1] == 1:
    sc = ax_grid.scatter(points[:, 0], obs_relerr, c=obs_relerr, cmap="Spectral_r")
    ax_grid.scatter(center[0], obs_relerr[chosen_idx], marker="x", s=120, c="black")
    ax_grid.set_xlabel("parameter 1")
    ax_grid.set_ylabel(f"relative error {chosen['observable_name']}")
else:
    varied_cols = list(VARIED_COLS)
    slice_idx = np.arange(len(points)) if points.shape[1] == 2 else nearest_slice(points, varied_cols, center, N_SLICE)
    xcol, ycol = varied_cols
    sc = ax_grid.scatter(points[slice_idx, xcol], points[slice_idx, ycol], c=obs_relerr[slice_idx], cmap="Spectral_r", s=64, edgecolor="black", linewidth=0.35)
    ax_grid.scatter(center[xcol], center[ycol], marker="x", s=130, c="black", linewidth=2)
    ax_grid.set_xlabel(f"parameter {xcol + 1}")
    ax_grid.set_ylabel(f"parameter {ycol + 1}")
    if points.shape[1] > 2:
        held = [col + 1 for col in range(points.shape[1]) if col not in varied_cols]
        ax_grid.set_title(f"nearest slice; held parameters {held} near chosen point")
    else:
        ax_grid.set_title("parameter grid")
fig.colorbar(sc, ax=ax_grid, label=f"relative error {chosen['observable_name']}")
ax_grid.grid(alpha=0.25)

ax_strength.plot(chosen["x"], chosen["y_true"], label="true", lw=1.7)
ax_strength.plot(chosen["x"], chosen["y_pred"], label="emulated", lw=1.9)
markerline, stemlines, _ = ax_strength.stem(chosen["poles"], chosen["strengths"], linefmt="0.55", markerfmt="o", basefmt=" ")
plt.setp(markerline, markersize=3, alpha=0.5)
plt.setp(stemlines, linewidth=0.8, alpha=0.3)
ax_strength.set_xlabel("Energy")
ax_strength.set_ylabel("Strength")
ax_strength.set_ylim(bottom=0)
ax_strength.set_title(
    "chosen point: " + ", ".join(f"p{i+1}={v:.4g}" for i, v in enumerate(chosen["params"]))
    + f"\nobs relerr={chosen['observable_relerr']:.3e}; strength L2={chosen['strength_rel_l2']:.3e}"
)
ax_strength.grid(alpha=0.25)
ax_strength.legend()
fig.suptitle(run["run_dir"].name)
fig.tight_layout()
display(fig)
plt.close(fig)


4. true vs emulated strength at the chosen point

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(chosen["x"], chosen["y_true"], label="true", lw=1.8)
ax.plot(chosen["x"], chosen["y_pred"], label="emulated", lw=1.9)
ax.set_xlabel("Energy")
ax.set_ylabel("Strength")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.25)
ax.legend()
ax.set_title("true vs emulated strength: " + ", ".join(f"p{i+1}={v:.4g}" for i, v in enumerate(chosen["params"])))
fig.tight_layout()
display(fig)
plt.close(fig)


5. parameter grid plus retained eigenvalue evolution along a sweep

In [ ]:
VARY_COL = 0  # 0 for p1, 1 for p2, etc.
N_SWEEP = 20

def choose_sweep(points, vary_col, center, n_sweep):
    held_cols = [col for col in range(points.shape[1]) if col != vary_col]
    if not held_cols:
        return np.argsort(points[:, vary_col])
    ranges = np.ptp(points[:, held_cols], axis=0)
    ranges = np.where(ranges > 0.0, ranges, 1.0)
    dist = np.linalg.norm((points[:, held_cols] - center[held_cols][None, :]) / ranges[None, :], axis=1)
    chosen_sweep = np.argsort(dist)[:min(n_sweep, len(points))]
    return chosen_sweep[np.argsort(points[chosen_sweep, vary_col])]

sweep_idx = choose_sweep(points, VARY_COL, center, N_SWEEP)
fig, (ax_grid, ax_eigs) = plt.subplots(1, 2, figsize=(13, 5.2))

if points.shape[1] == 1:
    sc = ax_grid.scatter(points[:, 0], obs_relerr, c=obs_relerr, cmap="Spectral_r")
    ax_grid.plot(points[sweep_idx, 0], obs_relerr[sweep_idx], "k.-")
    ax_grid.set_xlabel("parameter 1")
    ax_grid.set_ylabel("relative observable error")
else:
    xcol = 0 if VARY_COL != 0 else min(1, points.shape[1] - 1)
    ycol = VARY_COL
    sc = ax_grid.scatter(points[:, xcol], points[:, ycol], c=obs_relerr, cmap="Spectral_r", s=56)
    ax_grid.plot(points[sweep_idx, xcol], points[sweep_idx, ycol], "k.-", lw=1.1, ms=7)
    ax_grid.scatter(center[xcol], center[ycol], marker="x", s=130, c="black", linewidth=2)
    ax_grid.set_xlabel(f"parameter {xcol + 1}")
    ax_grid.set_ylabel(f"parameter {ycol + 1}")
fig.colorbar(sc, ax=ax_grid, label="relative observable error")
ax_grid.set_title("parameter grid and chosen sweep")
ax_grid.grid(alpha=0.25)

x = points[sweep_idx, VARY_COL]
max_modes = max(len(results[idx]["poles"]) for idx in sweep_idx)
eig_matrix = np.full((len(sweep_idx), max_modes), np.nan)
for row, idx in enumerate(sweep_idx):
    poles = np.asarray(results[idx]["poles"], dtype=float)
    eig_matrix[row, :len(poles)] = poles
for mode in range(eig_matrix.shape[1]):
    ax_eigs.plot(x, eig_matrix[:, mode], "o-", ms=3, lw=1.0, alpha=0.8)
ax_eigs.set_xlabel(f"parameter {VARY_COL + 1}")
ax_eigs.set_ylabel("retained eigenvalues")
ax_eigs.set_title("retained eigenvalue evolution")
ax_eigs.grid(alpha=0.25)
fig.suptitle(run["run_dir"].name)
fig.tight_layout()
display(fig)
plt.close(fig)
